# 06 — Target and Leakage Definition

## 1. Objective

Build the governed missed-resolution label and creation-time leakage policy without training a model.

In [1]:
from pathlib import Path
import sys
from IPython.display import Markdown, display
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
import pandas as pd

PROJECT_ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'src' / 'urban_ops').is_dir())
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from urban_ops.analysis.target_and_leakage import REQUIRED_REPORT_TABLES, build_step4_tables, write_step4_reports
from urban_ops.data.selected_scope import fetch_selected_scope_records, load_selected_scope_authority
from urban_ops.features.leakage import validate_feature_columns
from urban_ops.utils.paths import ensure_report_directories, notebook_report_paths

REPORT_DIR, TABLE_DIR, FIGURE_DIR = notebook_report_paths('06_target_and_leakage')
ensure_report_directories('06_target_and_leakage')
pd.set_option('display.max_columns', 80)


## 2. Authoritative selected scope

In [2]:
scope = load_selected_scope_authority()
scope_table = pd.DataFrame([{
    'agency': scope.agency, 'complaint_type': scope.complaint_type,
    'start_date': scope.start_date.date().isoformat(),
    'end_date': scope.end_date.date().isoformat(),
    'decision_status': scope.decision_status,
    'authority_extraction_timestamp': scope.authority_extraction_timestamp.isoformat(),
}])
display(scope_table.T)


,0
agency,DSNY
complaint_type,Graffiti
start_date,2024-01-01
end_date,2025-12-31
decision_status,APPROVED_WITH_LIMITATIONS
authority_extraction_timestamp,2026-07-27T17:04:43.549384+00:00


## 3. Extraction metadata and source snapshot

The live source is fetched with deterministic ordering. Counts are recomputed rather than copied from Step 3.

In [3]:
source, extraction_metadata = fetch_selected_scope_records(scope)
extraction_timestamp = pd.Timestamp(extraction_metadata.loc[0, 'extraction_timestamp'])
display(extraction_metadata.T)


,0
source,https://data.cityofnewyork.us/resource/erm2-nw...
dataset_identifier,erm2-nwe9
extraction_timestamp,2026-07-31T07:09:27.229097+00:00
scope_authority_extraction_timestamp,2026-07-27T17:04:43.549384+00:00
agency,DSNY
complaint_type,Graffiti
requested_start_date,2024-01-01
requested_end_date,2025-12-31
row_count,40017
ordering,"created_date ASC, unique_key ASC"


## 4. Required target fields

The reusable pipeline validates identifiers, scope fields, status, and all three target timestamps.

In [4]:
governed, tables = build_step4_tables(source, scope=scope, extraction_timestamp=extraction_timestamp)
write_step4_reports(tables, extraction_metadata, report_directory=REPORT_DIR)
display(governed[['unique_key', 'created_date', 'due_date', 'closed_date', 'status']].head())


,unique_key,created_date,due_date,closed_date,status
0,59892374,2024-01-01 03:03:35+00:00,NaT,2024-01-02 02:22:23+00:00,Closed
1,59894927,2024-01-01 07:58:15+00:00,2024-01-09 07:13:00+00:00,2024-02-08 02:22:23+00:00,Closed
2,59895663,2024-01-01 08:00:34+00:00,2024-01-31 08:00:34+00:00,2024-01-10 00:00:00+00:00,Closed
3,59894802,2024-01-01 08:01:53+00:00,2024-01-31 08:01:53+00:00,2024-02-29 00:00:00+00:00,Closed
4,59898230,2024-01-01 08:03:15+00:00,2024-01-09 07:13:00+00:00,2024-02-08 02:22:23+00:00,Closed


## 5. Status distribution

In [5]:
display(tables['status_analysis.csv'])


,status,row_count,row_share,created_date_coverage,due_date_coverage,closed_date_coverage,outcome_mature_count,target_eligible_count,proposed_treatment,final_treatment,decision_reason
0,closed,39740,0.993078,1.0,0.904882,1.00000,35960,35960,include when all other rules pass,include when all other rules pass,Closed is the only approved completed status.
1,open,271,0.006772,1.0,0.996310,0.00369,270,0,exclude,exclude,Not an approved completed operational outcome.
2,pending,6,0.000150,1.0,1.000000,1.00000,6,0,exclude,exclude,Not an approved completed operational outcome.


## 6. Missing target-input analysis

In [6]:
display(tables['eligibility_summary.csv'].loc[lambda x: x['rule_name'].isin(['has_created_date', 'has_due_date', 'has_closed_date'])])


,rule_name,pass_count,fail_count,pass_rate
1,has_created_date,40017,0,1.000000
2,has_due_date,36236,3781,0.905515
3,has_closed_date,39747,270,0.993253


## 7. Timestamp chronology analysis

In [7]:
display(tables['timestamp_violation_summary.csv'])


,violation,row_count
0,due_before_created,0
1,closed_before_created,6
2,status_closed_date_inconsistent,7


## 8. Open complaint analysis

Open complaints remain unlabeled; passing a due date does not prove eventual late closure.

In [8]:
display(governed.loc[governed['is_open'], ['status_normalized', 'outcome_mature', 'primary_exclusion_reason']].value_counts().rename('row_count').reset_index())


,status_normalized,outcome_mature,primary_exclusion_reason,row_count
0,open,True,missing_closed_date,269
1,pending,True,closed_before_created,6
2,open,True,excluded_status,1
3,open,False,missing_due_date,1


## 9. Cancelled complaint analysis

Cancellation is excluded because it is not an approved successful operational resolution.

In [9]:
display(governed.loc[governed['is_cancelled'], ['status_normalized', 'has_closed_date', 'primary_exclusion_reason']].value_counts().rename('row_count').reset_index())


,status_normalized,has_closed_date,primary_exclusion_reason,row_count


## 10. Duplicate analysis

In [10]:
display(tables['duplicate_summary.csv'])
display(tables['conflicting_duplicate_summary.csv'].head())


,metric,row_count
0,duplicate_unique_key_groups,0
1,rows_in_duplicate_groups,0
2,redundant_exact_duplicate_rows,0
3,conflicting_duplicate_groups,0
4,rows_in_conflicting_groups,0


,unique_key,row_count,conflicting_fields


## 11. Outcome maturity analysis

In [11]:
display(tables['outcome_maturity_summary.csv'])


,outcome_maturity,row_count,row_share
0,mature,36236,0.905515
1,not_mature,0,0.000000
2,missing_due_date,3781,0.094485
3,mature_open,276,0.006897


## 12. Final eligibility rules

Eligibility requires scope membership, valid required timestamps and chronology, approved closed status, a mature outcome, and an unambiguous canonical complaint ID.

In [12]:
display(tables['eligibility_summary.csv'])


,rule_name,pass_count,fail_count,pass_rate
0,within_selected_scope,40017,0,1.000000
1,has_created_date,40017,0,1.000000
2,has_due_date,36236,3781,0.905515
3,has_closed_date,39747,270,0.993253
4,valid_due_chronology,40017,0,1.000000
5,valid_closed_chronology,40011,6,0.999850
6,status_allowed,39740,277,0.993078
7,is_exact_duplicate,40017,0,1.000000
8,is_conflicting_duplicate,40017,0,1.000000
9,outcome_mature,36236,3781,0.905515


## 13. Target construction

`missed_resolution_target = closed_date > due_date`; equality is on time and ineligible rows remain nullable `NA`.

In [13]:
assert str(governed['missed_resolution_target'].dtype) == 'Int8'
assert governed.loc[~governed['target_eligible'], 'missed_resolution_target'].isna().all()


## 14. Target distribution

In [14]:
display(tables['target_distribution.csv'])


,target_value,target_label,row_count,row_share
0,0,on_time,20244,0.562959
1,1,missed,15716,0.437041


## 15. Exclusion-reason analysis

In [15]:
display(tables['exclusion_reason_summary.csv'])


,primary_exclusion_reason,row_count,row_share
0,eligible,35960,0.898618
1,missing_due_date,3781,0.094485
2,missing_closed_date,269,0.006722
3,closed_before_created,6,0.00015
4,excluded_status,1,0.000025


## 16. Leakage audit

In [16]:
validate_feature_columns(['created_date', 'agency', 'open_data_channel_type'])
display(tables['leakage_audit.csv'])


,source_column,internal_feature_name,data_type,role,available_at_creation,mutable_after_creation,allowed_for_baseline,null_policy,leakage_reason,decision_status,notes,creation_time_available,column_name,leakage_status,decision_reason
0,unique_key,unique_key,string,IDENTIFIER,True,False,False,preserve and handle explicitly,Identifier invites memorisation and is not a m...,BLOCKED,,True,unique_key,BLOCKED,Identifier invites memorisation and is not a m...
1,created_date,created_date,"datetime64[ns, UTC]",SAFE_FEATURE,True,False,True,preserve and handle explicitly,Available at creation; prefer derived calendar...,APPROVED,,True,created_date,SAFE,Available at creation; prefer derived calendar...
2,closed_date,closed_date,"datetime64[ns, UTC]",TARGET_INPUT,False,True,False,preserve and handle explicitly,Closure outcome is unavailable at complaint cr...,NOT_APPROVED,,False,closed_date,BLOCKED,Closure outcome is unavailable at complaint cr...
3,agency,agency,string,SAFE_FEATURE,True,False,True,preserve and handle explicitly,Available at complaint creation.,APPROVED,,True,agency,SAFE,Available at complaint creation.
4,agency_name,agency_name,string,SAFE_FEATURE,True,False,True,preserve and handle explicitly,Available at complaint creation.,APPROVED,,True,agency_name,SAFE,Available at complaint creation.
5,complaint_type,complaint_type,string,SAFE_FEATURE,True,False,True,preserve and handle explicitly,Available at complaint creation.,APPROVED,,True,complaint_type,SAFE,Available at complaint creation.
6,descriptor,descriptor,string,CONDITIONAL_FEATURE,None,None,False,preserve and handle explicitly,Intake-time availability or later correction i...,CONDITIONAL,,None,descriptor,CONDITIONAL,Intake-time availability or later correction i...
7,descriptor_2,descriptor_2,string,CONDITIONAL_FEATURE,None,None,False,preserve and handle explicitly,Intake-time availability or later correction i...,CONDITIONAL,,None,descriptor_2,CONDITIONAL,Intake-time availability or later correction i...
8,location_type,location_type,string,CONDITIONAL_FEATURE,None,None,False,preserve and handle explicitly,Intake-time availability or later correction i...,CONDITIONAL,,None,location_type,CONDITIONAL,Intake-time availability or later correction i...
9,incident_zip,incident_zip,string,CONDITIONAL_FEATURE,None,None,False,preserve and handle explicitly,Intake-time availability or later correction i...,CONDITIONAL,,None,incident_zip,CONDITIONAL,Intake-time availability or later correction i...


## 17. Feature-role classification

In [17]:
display(tables['feature_role_inventory.csv'])


,source_column,internal_feature_name,data_type,role,available_at_creation,mutable_after_creation,allowed_for_baseline,null_policy,leakage_reason,decision_status,notes,creation_time_available
0,unique_key,unique_key,string,IDENTIFIER,True,False,False,preserve and handle explicitly,Identifier invites memorisation and is not a m...,BLOCKED,,True
1,created_date,created_date,"datetime64[ns, UTC]",SAFE_FEATURE,True,False,True,preserve and handle explicitly,Available at creation; prefer derived calendar...,APPROVED,,True
2,closed_date,closed_date,"datetime64[ns, UTC]",TARGET_INPUT,False,True,False,preserve and handle explicitly,Closure outcome is unavailable at complaint cr...,NOT_APPROVED,,False
3,agency,agency,string,SAFE_FEATURE,True,False,True,preserve and handle explicitly,Available at complaint creation.,APPROVED,,True
4,agency_name,agency_name,string,SAFE_FEATURE,True,False,True,preserve and handle explicitly,Available at complaint creation.,APPROVED,,True
5,complaint_type,complaint_type,string,SAFE_FEATURE,True,False,True,preserve and handle explicitly,Available at complaint creation.,APPROVED,,True
6,descriptor,descriptor,string,CONDITIONAL_FEATURE,None,None,False,preserve and handle explicitly,Intake-time availability or later correction i...,CONDITIONAL,,None
7,descriptor_2,descriptor_2,string,CONDITIONAL_FEATURE,None,None,False,preserve and handle explicitly,Intake-time availability or later correction i...,CONDITIONAL,,None
8,location_type,location_type,string,CONDITIONAL_FEATURE,None,None,False,preserve and handle explicitly,Intake-time availability or later correction i...,CONDITIONAL,,None
9,incident_zip,incident_zip,string,CONDITIONAL_FEATURE,None,None,False,preserve and handle explicitly,Intake-time availability or later correction i...,CONDITIONAL,,None


## 18. Final governance decisions

Only creation-time safe fields pass by default. Conditional fields require approval. Due date is a target input and remains unapproved as a feature.

In [18]:
plots = [
    ('status_analysis.csv', 'status', 'row_share', 'status_distribution.png', 'Status distribution'),
    ('exclusion_reason_summary.csv', 'primary_exclusion_reason', 'row_share', 'exclusion_reason_distribution.png', 'Primary exclusions'),
    ('target_distribution.csv', 'target_label', 'row_share', 'target_distribution.png', 'Eligible target distribution'),
]
for filename, label, value, output, title in plots:
    plot = tables[filename]
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(plot[label].astype(str), plot[value], color='#4472C4')
    ax.yaxis.set_major_formatter(PercentFormatter(1.0)); ax.set_title(title)
    ax.tick_params(axis='x', rotation=35); fig.tight_layout()
    fig.savefig(FIGURE_DIR / output, dpi=160, bbox_inches='tight'); plt.close(fig)
waterfall = tables['eligibility_summary.csv']
fig, ax = plt.subplots(figsize=(10, 5)); ax.bar(waterfall['rule_name'], waterfall['pass_rate'], color='#70AD47')
ax.yaxis.set_major_formatter(PercentFormatter(1.0)); ax.set_title('Eligibility rule pass rates'); ax.tick_params(axis='x', rotation=55)
fig.tight_layout(); fig.savefig(FIGURE_DIR / 'eligibility_waterfall.png', dpi=160, bbox_inches='tight'); plt.close(fig)


## 19. Known limitations

Due-date creation timing, mutability, API version semantics, and operational meaning remain unproven. Conditional intake/geography fields also require lifecycle confirmation.

## 20. Step 4 completion summary

In [19]:
summary = tables['target_summary.csv']
assert len(summary) == 1
assert summary.loc[0, 'total_selected_rows'] == len(governed)
assert summary.loc[0, 'target_eligible_rows'] == governed['target_eligible'].sum()
assert not governed.loc[governed['target_eligible'], 'unique_key'].duplicated().any()
assert all((TABLE_DIR / name).is_file() for name in REQUIRED_REPORT_TABLES)
assert all((FIGURE_DIR / name).is_file() for name in ['status_distribution.png', 'exclusion_reason_distribution.png', 'target_distribution.png', 'eligibility_waterfall.png'])
display(summary.T)
display(Markdown('**Step 4 target and leakage governance completed for the authoritative selected scope.**'))


,0
selected_agency,DSNY
selected_complaint_type,Graffiti
selected_start_date,2024-01-01
selected_end_date,2025-12-31
extraction_timestamp,2026-07-31T07:09:27.229097+00:00
total_selected_rows,40017
target_eligible_rows,35960
target_ineligible_rows,4057
eligibility_rate,0.898618
missed_count,15716


**Step 4 target and leakage governance completed for the authoritative selected scope.**